In [7]:
!pip install openai
!pip install dotenv

In [ ]:
import base64
import json
from openai import OpenAI
import os
from dotenv import load_dotenv

# Load variables from .env into the environment
load_dotenv()

# Access the API key
openai_api_key = os.getenv("OPENAI_API_KEY")

# Use the key in your application
# print(f"Key loaded: {openai_api_key}")

client = OpenAI(api_key=openai_api_key)  # or set OPENAI_API_KEY env var

In [2]:
def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [ ]:
def extract_text_from_image(image_path: str, model: str = "gpt-4o") -> dict:
    base64_image = encode_image(image_path)
    ext = image_path.split(".")[-1].lower()
    mime = "jpeg" if ext in ["jpg", "jpeg"] else ext

    system_prompt = (
        "You are an OCR assistant. Extract ALL visible text from the image. "
        "Return ONLY valid JSON, with no markdown formatting or code fences, "
        "matching this schema:\n"
        "{\n"
        '  "full_text": "all text concatenated in reading order",\n'
        '  "text_blocks": [\n'
        '    {"text": "block of text", "location": "brief description, e.g. top-left, header, table"}\n'
        "  ]\n"
        "}"
    )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Extract the text from this image."},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/{mime};base64,{base64_image}"
                        },
                    },
                ],
            },
        ],
        max_tokens=2000,
        temperature=0,
    )

    raw_content = response.choices[0].message.content.strip()

    # Strip accidental code fences if the model adds them
    if raw_content.startswith("```"):
        raw_content = raw_content.strip("`")
        raw_content = raw_content.replace("json\n", "", 1).strip()

    try:
        return json.loads(raw_content)
    except json.JSONDecodeError:
        return {"error": "Failed to parse JSON", "raw_response": raw_content}

In [6]:
image_path = "bulletin.jpg"
result = extract_text_from_image(image_path)
print(json.dumps(result, indent=2))

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
with open("extracted_text.json", "w") as f:
    json.dump(result, f, indent=2)